# Exploratory Dataset Analysis (EDA) on the CityScapes Image Dataset

### Overview
- 34 classes, we used 19*
- About 50 cities across Germany and Switzerland, we used 21*
- Several months/times of year (spring, summer, autumn)
- Good and moderate weather conditions

### Our Data
- Original image resolution: 1024x2048
- Total number of images: 3475
- Split:
    - train: 15 cities - 2439 images (~70%)
    - validation: 3 cities - 500 images (~15%)
    - test: 3 cities - 536 images (~15%)


In [ ]:
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm.auto import tqdm

In [ ]:
sns.set_theme(style="whitegrid")
DATASET_DIR = Path("./data/dataset")
FIGURE_DIR = Path("./eda/figures")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "val", "test"]
NUM_CLASSES = 19    # cityscapes has 30 classes in total but only evaluates 19 of them
IGNORE_INDEX = 255  # pixels to ignore

In [ ]:
# function to save figures and plots
def save_figure(filename, dpi=300):
    path = FIGURE_DIR / filename

    plt.savefig(
        path,
        dpi=dpi,
        bbox_inches="tight",
        facecolor="white"
    )

    print(f"Saved: {path}")

In [ ]:
# the 19 classes mainly used in cityscapes image segmentation
CLASS_NAMES = [
    "road", "sidewalk", "building", "wall", "fence",
    "pole", "traffic light", "traffic sign",
    "vegetation", "terrain", "sky", "person",
    "rider", "car", "truck", "bus", "train",
    "motorcycle", "bicycle"
]

CLASS_COLORS = np.array([
    [128, 64, 128],         # road = purple
    [244, 35, 232],         # sidewalk = pink/magenta
    [70, 70, 70],           # building = dark grey
    [102, 102, 156],        # wall = greyish purple
    [190, 153, 153],        # fence = dusty pink
    [153, 153, 153],        # pole = light grey
    [250, 170, 30],         # traffic light = orange
    [220, 220, 0],          # traffic sign = yellow
    [107, 142, 35],         # vegetation = olive green
    [152, 251, 152],        # terrain = light green
    [70, 130, 180],         # sky = steel blue
    [220, 20, 60],          # person = crimson red
    [255, 0, 0],            # rider = bright red
    [0, 0, 142],            # car = dark blue
    [0, 0, 70],             # truck = very dark navy blue
    [0, 60, 100],           # bus = dark teal blue
    [0, 80, 100],           # train = dark teal
    [0, 0, 230],            # motorcycle = vivid blue
    [119, 11, 32]           # bicycle = dark burgundy red
], dtype=np.uint8)

# original labels in the order of the class names above
LABEL_IDS = [
    7, 8, 11, 12, 13, 17, 19, 20, 21,
    22, 23, 24, 25, 26, 27, 28, 31, 32, 33
]

# mapping from original label IDs to train IDs (0-18)
LABEL_TO_TRAIN = {
    label_id: train_id
    for train_id, label_id in enumerate(LABEL_IDS)
}

# lookup table mapping original IDs to train IDs
ID_LOOKUP = np.full(256, IGNORE_INDEX, dtype=np.uint8)

# replaces the lookup values with the 19 chosen classes
for label_id, train_id in LABEL_TO_TRAIN.items():
    ID_LOOKUP[label_id] = train_id

In [ ]:
# converts the masks with the original label IDs to masks with the train IDs (0-18) and ignores the rest
def convert_to_train_ids(mask):
    mask = np.asarray(mask)

    if mask.ndim != 2:
        raise ValueError("Expected a 2D labelIds mask")
    if not np.issubdtype(mask.dtype, np.integer):
        raise ValueError("Mask must contain integer label IDs")

    result = np.full(mask.shape, IGNORE_INDEX, dtype=np.uint8)
    valid = (mask >= 0) & (mask < 256)
    result[valid] = ID_LOOKUP[mask[valid].astype(np.int64)]

    return result

In [ ]:
# check for missing masks and create a dataframe with image and mask paths
records = []

for split in SPLITS:
    split_dir = DATASET_DIR / split

    image_paths = sorted(split_dir.glob("*_leftImg8bit.png"))

    for image_path in image_paths:
        base_name = image_path.name.replace("_leftImg8bit.png", "")
        mask_path = split_dir / (base_name + "_gtFine_labelIds.png")
        city = base_name.split("_")[0]

        records.append({
            "split": split,
            "city": city,
            "image_name": image_path.name,
            "image_path": str(image_path),
            "mask_path": str(mask_path),
            "mask_exists": mask_path.exists()
        })

df = pd.DataFrame(records)

print("Total images:", len(df))
print("Missing masks:", (~df["mask_exists"]).sum())

display(df.head())

In [ ]:
missing = df[~df["mask_exists"]]
display(missing)

for split in SPLITS:
    split_dir = DATASET_DIR / split

    images = list(split_dir.glob("*_leftImg8bit.png"))
    masks = list(split_dir.glob("*_gtFine_labelIds.png"))

    print(split, "| Images:", len(images), "| Masks:", len(masks))

All images have a corresponding mask.

In [ ]:
# calculate training/validation/test split counts and percentages
split_counts = (
    df.groupby("split")
      .size()
      .reindex(SPLITS, fill_value=0)
)

split_percentages = (100 * split_counts / split_counts.sum())

split_summary = pd.DataFrame({
    "Images": split_counts,
    "Percentage": split_percentages.round(2)
})

display(split_summary)

In [ ]:
# visualise the distribution of images across splits
plt.figure(figsize=(8, 5))
sns.barplot(x=split_counts.index, y=split_counts.values)

plt.title("Cityscapes Dataset Split Distribution")
plt.xlabel("Dataset Split")
plt.ylabel("Number of Images")
plt.tight_layout()
save_figure("dataset_split_distribution.png")
plt.show()

In [ ]:
# display the number of images per city for each split
city_counts = (df.groupby(["split", "city"]).size().reset_index(name="image_count"))

plt.figure(figsize=(14, 6))
sns.barplot(
    data=city_counts,
    x="city",
    y="image_count",
    hue="split"
)

plt.title("Number of Images per City")
plt.xlabel("City")
plt.ylabel("Number of Images")
plt.xticks(rotation=65, ha="right")
plt.tight_layout()
save_figure("num_images_per_city.png")
plt.show()

city_stats = (
    city_counts
    .groupby("split")["image_count"]
    .agg(
        city_count="count",
        total_images="sum",
        mean="mean",
        std="std",
        minimum="min",
        q1=lambda x: x.quantile(0.25),
        median="median",
        q3=lambda x: x.quantile(0.75),
        maximum="max"
    )
    .round(2)
)

display(city_stats)

Cities like Strasbourg and Bremen have significantly more images than cities like Bochum and Darmstadt. This means that the cities with more images will have a greater influence on the model's learning. This could matter if the city with more images has a very different street and road structure than other cities.

In [ ]:
# check for overlapping cities between splits to identify any data leakage
split_cities = {
    split: set(df.loc[df["split"] == split, "city"])
    for split in SPLITS
}

for i, split_a in enumerate(SPLITS):
    for split_b in SPLITS[i + 1:]:
        overlap = split_cities[split_a] & split_cities[split_b]

        print(
            f"{split_a} vs {split_b}:",
            overlap if overlap else "No city overlap"
        )

There is no overlap of images between different cities, so choosing to split our training, validation, and test sets by city ensures no data leakage. If there was an overlap between training and test cities, the model may remember an image in the test set from the training and apply the previous training mask rather than identifying and segmenting the image from scratch.

In [ ]:
# gather image metadata such as dimensions and color mode for both images and masks
# allows us to make sure all images are the same size and have the same number of input channels (3 for RGB)
metadata = []

for row in tqdm(df.itertuples(index=False), total=len(df)):
    if not row.mask_exists:
        continue

    try:
        with Image.open(row.image_path) as image:
            width, height = image.size
            image_mode = image.mode

        with Image.open(row.mask_path) as mask:
            mask_width, mask_height = mask.size
            mask_mode = mask.mode

        metadata.append({
            "split": row.split,
            "city": row.city,
            "image_path": row.image_path,
            "mask_path": row.mask_path,
            "width": width,
            "height": height,
            "image_mode": image_mode,
            "mask_mode": mask_mode,
            "dimensions_match": (
                width == mask_width and
                height == mask_height
            )
        })

    except Exception as e:
        print("Error:", row.image_path, e)

meta_df = pd.DataFrame(metadata)

display(meta_df.head())
print(meta_df[["width", "height"]].describe())
print("Dimension mismatches:",
      (~meta_df["dimensions_match"]).sum())

In [ ]:
# image width and height distributions (uninteresting as all images are the same size)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(meta_df["width"], ax=axes[0])
axes[0].set_title("Image Width Distribution")

sns.histplot(meta_df["height"], ax=axes[1])
axes[1].set_title("Image Height Distribution")

plt.tight_layout()
save_figure("image_dimensions_distribution.png")
plt.show()

All images and masks have the same resolution (2048x1024) which means we do not have to apply any cropping or resizing as part of out pre-processing. We will keep the original resolution and maybe crop or resize during training depending on the training speed and need for data augmentation.

**Tip from a friend:** "Always use bilinear interpolation for RGB images and nearest-neighbour interpolation for class-ID masks."

In [ ]:
# add colour to the greyscale masks for better visualisation
def colorize_mask(train_mask):
    h, w = train_mask.shape
    colored = np.zeros((h, w, 3), dtype=np.uint8)

    for class_id, color in enumerate(CLASS_COLORS):
        colored[train_mask == class_id] = color

    return colored

In [ ]:
# display a sample image, its corresponding segmentation mask, and an overlay of the two
def visualize_sample(row):
    image = np.asarray(Image.open(row["image_path"]).convert("RGB"))
    original_mask = np.asarray(Image.open(row["mask_path"]))
    train_mask = convert_to_train_ids(original_mask)
    color_mask = colorize_mask(train_mask)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(image)
    axes[0].set_title("Original RGB Image")

    axes[1].imshow(color_mask)
    axes[1].set_title("19-Class Segmentation Mask")

    axes[2].imshow(image)
    axes[2].imshow(color_mask, alpha=0.5)
    axes[2].set_title("Segmentation Overlay")

    for ax in axes:
        ax.axis("off")

    plt.suptitle(f"City: {row['city']} | Split: {row['split']}")
    plt.tight_layout()
    plt.show()

In [ ]:
for split in SPLITS:
    sample = df[
        (df["split"] == split) &
        (df["mask_exists"])
    ].sample(1, random_state=42).iloc[0]

    visualize_sample(sample)

All images and masks align correctly and the segmentation is highly accurate. A slight setback is when multiple vehicles or vegetation are close together in the image, they are segmented as one group in the mask. The labels appear consistent across the three samples, i.e. cars are blue, vegetation is green, humans are red, etc.

In [ ]:
# inspect the mask IDs and compare against the 19 chosen classes
raw_label_counts = Counter()

for mask_path in tqdm(meta_df["mask_path"]):
    mask = np.asarray(Image.open(mask_path))
    labels, counts = np.unique(mask, return_counts=True)

    raw_label_counts.update({
        int(label): int(count)
        for label, count in zip(labels, counts)
    })

raw_label_df = pd.DataFrame([
    {"label_id": k, "pixel_count": v}
    for k, v in raw_label_counts.items()
])

raw_label_df["train_id"] = (
    raw_label_df["label_id"]
    .map(LABEL_TO_TRAIN)
    .fillna(IGNORE_INDEX)
    .astype(int)
)

raw_label_df["status"] = np.where(
    raw_label_df["train_id"] == IGNORE_INDEX,
    "Ignored",
    "Training class"
)

raw_label_df["class_name"] = raw_label_df["train_id"].apply(
    lambda x: CLASS_NAMES[x]
    if x != IGNORE_INDEX else "Ignored"
)

display(raw_label_df.sort_values("pixel_count", ascending=False))

We see the CityScapes dataset's original 34 classes, of which we use 19 and map the rest to 255 (ignored). As expected, road, buildings, and vegetation cover the largest number of pixels across all images in the dataset. Buses, trains, traffic lights, riders, and motorcycles are among the smallest.

In [ ]:
# analyse class distribution and check for imbalance
pixel_records = []
image_class_records = []

for row in tqdm(meta_df.itertuples(index=False), total=len(meta_df)):
    original_mask = np.asarray(Image.open(row.mask_path))
    train_mask = convert_to_train_ids(original_mask)
    counts = np.bincount(train_mask.ravel(), minlength=256)
    total_pixels = train_mask.size

    record = {
        "split": row.split,
        "city": row.city,
        "image_path": row.image_path,
        "total_pixels": total_pixels,
        "ignored_pixels": int(counts[255])
    }

    for class_id, class_name in enumerate(CLASS_NAMES):
        count = int(counts[class_id])
        record[class_name] = count

        image_class_records.append({
            "split": row.split,
            "city": row.city,
            "image_path": row.image_path,
            "class_name": class_name,
            "pixel_count": count,
            "present": count > 0,
            "area_percentage": 100 * count / total_pixels
        })

    pixel_records.append(record)

pixel_df = pd.DataFrame(pixel_records)  # one row per image with pixel counts for each class
image_class_df = pd.DataFrame(image_class_records) # one row per image/class combination with class presence and area percentage

In [ ]:
# global class distribution across the entire dataset
# analyses the amount of space each class occupies in the dataset (how many pixels it covers)
class_totals = (
    pixel_df[CLASS_NAMES]
    .sum()
    .sort_values(ascending=False)
)

total_valid_pixels = class_totals.sum()
class_percentages = (
    class_totals / total_valid_pixels * 100
)

plt.figure(figsize=(14, 6))
sns.barplot(
    x=class_percentages.index,
    y=class_percentages.values
)

plt.title("Pixel Distribution Across 19 Cityscapes Classes")
plt.xlabel("Semantic Class")
plt.ylabel("Percentage of Valid Pixels")
plt.xticks(rotation=65, ha="right")
plt.tight_layout()
save_figure("pixel_distribution_by_class.png")
plt.show()

Very severe class imbalance, which was expected. Over 35% of the pixels across all images in the dataset contain road, then about 23% take up buildings, and about 16% make up vegetation. Classes like bus, train, traffic light, rider, and motorcycle take up a very small amount of pixels, seemingly less than 1% each. This imbalance may bias the model's optimisation towards the more densely represented classes.

**Potential Solutions:** Class-aware sampling and weighted loss functions such as weighted cross-entropy.

In [ ]:
# image-level class frequency distribution (how many images contain each class)
class_presence = (
    image_class_df
    .groupby("class_name")["present"]
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

plt.figure(figsize=(14, 6))
sns.barplot(
    x=class_presence.index,
    y=class_presence.values
)

plt.title("Percentage of Images Containing Each Class")
plt.ylabel("Images Containing Class (%)")
plt.xlabel("Class")
plt.xticks(rotation=65, ha="right")
plt.tight_layout()
save_figure("image-level_class_distribution.png")
plt.show()

Despite the severe imbalance in classes, each class does appear in at least 1% of the total images in the dataset. Common features such as poles, buildings, roads, vegetation, cars, traffic signs, and sidewalks appear in over 90% of images, while motorcycles, trucks, buses, adn trains appear in less than 20%.

For classes that appear frequently but occupy a small number of pixels, it's recommended to try to preserve the resolution.  
For classes that are both rare and take up little space, we should consider targeted sampling, but avoid oversampling so that the model doesn't memorise a small number of examples.

In [ ]:
# analyse ignored pixels
pixel_df["ignored_percentage"] = (
    100 * pixel_df["ignored_pixels"]
    / pixel_df["total_pixels"]
)

sns.boxplot(
    data=pixel_df,
    x="split",
    y="ignored_percentage",
    order=SPLITS
)

plt.title("Ignored Pixel Percentage by Dataset Split")
plt.ylabel("Ignored Pixels (%)")
save_figure("ignored_pixel_distribution.png")
plt.show()

In [ ]:
# class distribution across training, validation, and test splits
split_class_counts = (
    pixel_df.groupby("split")[CLASS_NAMES]
    .sum()
    .reindex(SPLITS)
)

split_class_percentages = (
    split_class_counts.div(
        split_class_counts.sum(axis=1),
        axis=0
    ) * 100
)

plt.figure(figsize=(16, 5))
sns.heatmap(
    split_class_percentages,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu"
)

plt.title("Class Pixel Distribution Across Dataset Splits")
plt.xlabel("Class")
plt.ylabel("Split")
plt.tight_layout()
save_figure("class_distribution_by_split.png")
plt.show()

The class distribution between splits is very even, so the splits are well-representative of each class. No classes appear much more often in training than in validation or testing, so the model should be trained well over all classes and so should perform well on unseen images.

In [ ]:
# class distribution by city
city_class_counts = (
    pixel_df.groupby(["split", "city"])[CLASS_NAMES]
    .sum()
)

city_class_percentages = (
    city_class_counts.div(
        city_class_counts.sum(axis=1),
        axis=0
    ) * 100
)

plt.figure(figsize=(18, 10))

sns.heatmap(
    city_class_percentages,
    cmap="viridis",
    linewidths=0.3
)

plt.title("Class Distribution Across Cityscapes Cities")
plt.xlabel("Semantic Class")
plt.ylabel("Dataset Split / City")
plt.tight_layout()
save_figure("class_distribution_by_city.png")
plt.show()

The class distribution between cities is also very uniform, meaning it's good and balanced to keep our splits separated by city. No one city has a much denser presence of any particular class than another.

In [ ]:
# number of classes per image
pixel_df["num_classes_present"] = (
    pixel_df[CLASS_NAMES] > 0
).sum(axis=1)

plt.figure(figsize=(10, 5))

sns.histplot(
    data=pixel_df,
    x="num_classes_present",
    hue="split",
    discrete=True,
    multiple="dodge"
)

plt.title("Number of Semantic Classes per Image")
plt.xlabel("Number of Classes Present")
plt.ylabel("Number of Images")
plt.tight_layout()
save_figure("num_classes_per_image.png")
plt.show()

This is generally a good distribution. Most images contain between 9 and 15 different classes, with only a few simple outlier images. These simple images are still valid and good quality, so we'll keep them in our dataset to represent variation in the complexity of different cities.

In [ ]:
# class diversity by city
plt.figure(figsize=(15, 6))

sns.boxplot(
    data=pixel_df,
    x="city",
    y="num_classes_present",
    hue="split"
)

plt.title("Class Diversity per Image Across Cities")
plt.xticks(rotation=65, ha="right")
plt.tight_layout()
save_figure("class_diversity_by_city.png")
plt.show()

Most cities have similar image-level class diversity, but Lindau stands out with significantly lower class diversity. From previous visuals, we saw that Lindau contains the least amount of images out of each city in the dataset, although this doesn't necessarily explain the lower number of classes per image. This is generally a sign of a different scene composition and a different type of urban envirnoment than what is typically seen in the other cities. We decided to keep it in our split to once again preserve the range of visual conditions represented in the dataset.

In [ ]:
# identify rare classes and which images contain them
# 5 rarest classes (wall, motorcycle, truck, bus, train)
rare_classes = class_presence[len(class_presence)-5:].index.tolist()

for class_name in rare_classes:
    subset = image_class_df[
        (image_class_df["class_name"] == class_name) &
        (image_class_df["present"])
    ]

    print(
        class_name,
        "| Images containing class:",
        len(subset)
    )

    display(
        subset.groupby(["split", "city"])
        .size()
        .rename("images")
        .reset_index()
    )

We took the five rarest classes (wall, motorcycle, bus, truck, and train) and analysed how many images each of them appeared in. Even the rarest class was present in over 150 images.

In [ ]:
# find images with the largest pixel areas for a particular class
target_class = "train"

examples = (
    image_class_df[
        (image_class_df["class_name"] == target_class) &
        (image_class_df["area_percentage"] > 0)
    ]
    .sort_values("area_percentage", ascending=False)
    .head(5)
)

for path in examples["image_path"]:
    row = df[df["image_path"] == path].iloc[0]
    visualize_sample(row)

It was good to see that the rarest class (train) covered a large percentage of pixels in some of the images it appeared in. However, there are many images where this class takes up less than 1% of the area in the image. For the "train" class, there are 64 images where trains occupy between 0.1% and 1.0% of the image. The solution would be to use higher-resolution crops and class-aware sampling if we want to focus on these rarer classes.

In [ ]:
# find images with the smallest non-zero pixel areas for a particular class
target_class = "train"

# Find all images where the target class occupies between 0.1% and 1.0%
filtered_images = image_class_df[
    (image_class_df["class_name"] == target_class) &
    (image_class_df["area_percentage"] > 0.1) &
    (image_class_df["area_percentage"] < 1.0)
]

print(f"Number of images in range: {len(filtered_images)}")

# Display the 5 images with the smallest areas in this range
examples = (
    filtered_images
    .sort_values("area_percentage", ascending=True)
    .head(5)
)

for path in examples["image_path"]:
    row = df[df["image_path"] == path].iloc[0]
    visualize_sample(row)

In [ ]:
# analyse image brightness and RGB distributions
rgb_records = []

for row in tqdm(meta_df.itertuples(index=False),
                total=len(meta_df)):

    with Image.open(row.image_path) as im:
        image = np.asarray(
            im.convert("RGB").resize((256, 128))
        )

    rgb_records.append({
        "split": row.split,
        "city": row.city,
        "mean_red": image[:, :, 0].mean(),
        "mean_green": image[:, :, 1].mean(),
        "mean_blue": image[:, :, 2].mean(),
        "brightness": image.mean(),
        "contrast": image.std()
    })

rgb_df = pd.DataFrame(rgb_records)

In [ ]:
# plot brightness by city
plt.figure(figsize=(15, 6))

sns.boxplot(
    data=rgb_df,
    x="city",
    y="brightness",
    hue="split"
)

plt.title("Image Brightness Across Cityscapes Cities")
plt.xticks(rotation=65, ha="right")
plt.tight_layout()
save_figure("image_brightness_by_city.png")
plt.show()

There is some expected variance between image brightness in different cities, which may be down to the time of day or weather when the image groups were taken, and also due to the colour of the architecture in different cities. The variance is not excessive and there are no major outliers, so we kept the data as is. We could therefor use standard RBG normalisation.

In [ ]:
# plot contrast by city
plt.figure(figsize=(15, 6))

sns.boxplot(
    data=rgb_df,
    x="city",
    y="contrast",
    hue="split"
)

plt.title("Image Contrast Across Cities")
plt.xticks(rotation=65, ha="right")
plt.tight_layout()
save_figure("image_contrast_by_city.png")
plt.show()

As with image brightness, there is some variance in image contrast between cities, but no major outliers, so standard RBG normalisation would suffice.

In [ ]:
# class co-occurrence analysis (which classes tend to appear together in the same image)
presence_matrix = (
    pixel_df[CLASS_NAMES] > 0
).astype(int)

cooccurrence = (
    presence_matrix.T @ presence_matrix
)

plt.figure(figsize=(13, 11))

sns.heatmap(
    cooccurrence,
    cmap="Blues"
)

plt.title("Semantic Class Co-occurrence Matrix")
plt.xlabel("Class")
plt.ylabel("Class")
plt.tight_layout()
save_figure("class_cooccurrence_matrix.png")
plt.show()

We identified a high co-occurance between classes like road, sidewalk, building, pole, traffic sign, vegeation, car, and sky, which was expected as these object often appear together in city streetviews and these are among the more common classes in the dataset. We also noticed that the rare classes, namely truck, bus, train, and motorcycle, do not appear frequently with many other classes at all.

**Potential Solution**: Targeted sampling as previously mentioned.

In [ ]:
# validate split integrity and check duplicates
import hashlib

def calculate_hash(path):
    hasher = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            hasher.update(chunk)

    return hasher.hexdigest()

hash_records = []

for row in tqdm(meta_df.itertuples(index=False),
                total=len(meta_df)):

    hash_records.append({
        "split": row.split,
        "city": row.city,
        "image_path": row.image_path,
        "hash": calculate_hash(row.image_path)
    })

hash_df = pd.DataFrame(hash_records)

duplicates = hash_df[
    hash_df.duplicated("hash", keep=False)
]

display(duplicates.sort_values("hash"))

We confirmed once again that there was no image overlap, duplication, or data leakages between our training, validation, and test sets.

In [ ]:
# save tables for training
OUTPUT_DIR = Path("./eda/tables")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT_DIR / "dataset_inventory.csv", index=False)
meta_df.to_csv(OUTPUT_DIR / "image_metadata.csv", index=False)
pixel_df.to_csv(OUTPUT_DIR / "pixel_statistics.csv", index=False)
image_class_df.to_csv(OUTPUT_DIR / "image_class_statistics.csv", index=False)
city_class_percentages.to_csv(OUTPUT_DIR / "city_class_distribution.csv")
split_class_percentages.to_csv(OUTPUT_DIR / "split_class_distribution.csv")
rgb_df.to_csv(OUTPUT_DIR / "rgb_statistics.csv", index=False)

print("EDA results saved.")

### NOTE: Save these for the report
| Figure | Visualisation | Purpose | Included? |
|--|--|--|--|
| 1 | Images per dataset split | Show dataset allocation | ✅ |
| 2 | Images per city | Examine city imbalance | ✅ |
| 3 | RGB image, colour mask and overlay | Verify segmentation labels | ✅ |
| 4 | Pixel frequency of 19 classes | Identify class imbalance | ✅ |
| 5 | Image-level class frequency | Identify class prevalence | ✅ |
| 6 | Class distribution by city | Investigate geographic domain shift | ✅ |
| 7 | Class distribution by split | Assess train/test representativeness | ✅ |
| 8 | Classes per image | Measure semantic diversity | ✅ |
| 9 | Brightness and contrast by city | Explore appearance differences | ✅ |
|10 | Class co-occurrence heatmap | Understand relationships between labels | ✅ |

# HOW TO INTERPRET